In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# import ...
import math
import sys
import matplotlib.pyplot as plt

# from ... import ...
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from IPython.display import HTML, display

# Get Path
from pathlib import Path

# Get
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# from ... import ...
from src import (load_profile_bundle, multimode_deformed_wing_surface, run_profile_experiment, wing_surface_mesh)

In [ ]:
def mesh_faces_mm(mesh):
    return [
        [
            (mesh.x_m[row][column] * 1e3, mesh.y_m[row][column] * 1e3, mesh.z_m[row][column] * 1e3),
            (mesh.x_m[row + 1][column] * 1e3, mesh.y_m[row + 1][column] * 1e3, mesh.z_m[row + 1][column] * 1e3),
            (mesh.x_m[row + 1][column + 1] * 1e3, mesh.y_m[row + 1][column + 1] * 1e3, mesh.z_m[row + 1][column + 1] * 1e3),
            (mesh.x_m[row][column + 1] * 1e3, mesh.y_m[row][column + 1] * 1e3, mesh.z_m[row][column + 1] * 1e3),
        ]
        for row in range(len(mesh.x_m) - 1)
        for column in range(len(mesh.x_m[row]) - 1)
    ]


def source_module_faces_mm(bundle):
    def add(first, second):
        return tuple(first[index] + second[index] for index in range(3))

    def scale(vector, factor):
        return tuple(component * factor for component in vector)

    def dot(first, second):
        return sum(first[index] * second[index] for index in range(3))

    def cross(first, second):
        return (
            first[1] * second[2] - first[2] * second[1],
            first[2] * second[0] - first[0] * second[2],
            first[0] * second[1] - first[1] * second[0],
        )

    def unit(vector):
        magnitude = math.sqrt(dot(vector, vector))
        return scale(vector, 1.0 / magnitude)

    def ring(center, radius, first_axis, second_axis, count=24):
        return [
            add(
                center,
                add(
                    scale(first_axis, radius * math.cos(2.0 * math.pi * index / count)),
                    scale(second_axis, radius * math.sin(2.0 * math.pi * index / count)),
                ),
            )
            for index in range(count)
        ]

    def cylinder_faces(start, stop, radius, first_axis, second_axis):
        start_ring = ring(start, radius, first_axis, second_axis)
        stop_ring = ring(stop, radius, first_axis, second_axis)
        sides = [
            [start_ring[index], start_ring[(index + 1) % len(start_ring)],
             stop_ring[(index + 1) % len(stop_ring)], stop_ring[index]]
            for index in range(len(start_ring))
        ]
        return [start_ring, list(reversed(stop_ring)), *sides]

    def box_faces(start, stop, half_width, half_height, first_axis, second_axis):
        def corners(center):
            return [
                add(center, add(scale(first_axis, width_sign * half_width),
                                scale(second_axis, height_sign * half_height)))
                for width_sign, height_sign in ((-1.0, -1.0), (1.0, -1.0),
                                                  (1.0, 1.0), (-1.0, 1.0))
            ]

        first = corners(start)
        second = corners(stop)
        sides = [
            [first[index], first[(index + 1) % 4],
             second[(index + 1) % 4], second[index]]
            for index in range(4)
        ]
        return [first, list(reversed(second)), *sides]

    acoustics = bundle.profile.acoustics
    source = bundle.source_position_m
    outward = unit(tuple(source[index] - bundle.output_position_m[index] for index in range(3)))
    reference = (1.0, 0.0, 0.0) if abs(outward[0]) < 0.9 else (0.0, 1.0, 0.0)
    first_axis = unit(tuple(reference[index] - dot(reference, outward) * outward[index]
                            for index in range(3)))
    second_axis = unit(cross(outward, first_axis))
    buzzer_stop = add(source, scale(outward, acoustics.pcb_buzzer_size_height_mm * 1e-3))
    module_stop = add(source, scale(outward, acoustics.pcb_size_height_mm * 1e-3))
    buzzer_faces = cylinder_faces(
        source, buzzer_stop, acoustics.pcb_buzzer_size_radius_mm * 1e-3,
        first_axis, second_axis,
    )
    pcb_faces = box_faces(
        buzzer_stop, module_stop, 0.5 * acoustics.pcb_size_horizontal_mm * 1e-3,
        0.5 * acoustics.pcb_size_vertical_mm * 1e-3, first_axis, second_axis,
    )
    points = [point for faces in (buzzer_faces, pcb_faces) for face in faces for point in face]

    def to_mm(faces):
        return [[tuple(component * 1e3 for component in point) for point in face] for face in faces]

    return to_mm(buzzer_faces), to_mm(pcb_faces), [
        tuple(component * 1e3 for component in point) for point in points
    ]

In [ ]:
def animate_selected_wing_3d(bundle, result, deformation_scale=100.0, frame_count=80, interval_ms=33):
    mesh = wing_surface_mesh(bundle.wing, n_span=51, n_chord=11)
    output = tuple(value * 1e3 for value in bundle.output_position_m)
    source = tuple(value * 1e3 for value in bundle.source_position_m)
    buzzer_faces, pcb_faces, module_points = source_module_faces_mm(bundle)
    root = (
        0.0,
        0.5 * (min(mesh.y_m[0]) + max(mesh.y_m[0])) * 1e3,
        0.5 * (min(mesh.z_m[0]) + max(mesh.z_m[0])) * 1e3,
    )
    body_arrow_length_mm = max(0.18 * bundle.wing.length_m * 1e3, 3.0)
    body_label_x = -1.15 * body_arrow_length_mm
    maximum_deformation_mm = deformation_scale * 1e3 * max(
        max(abs(value) for value in result.tip_displacement_m),
        max(abs(value) for value in result.output_displacement_m),
    )

    coordinates = (
        [value * 1e3 for row in mesh.x_m for value in row]
        + [output[0], source[0], body_label_x]
        + [point[0] for point in module_points],
        [value * 1e3 for row in mesh.y_m for value in row]
        + [output[1], source[1], root[1]]
        + [point[1] for point in module_points],
        [value * 1e3 for row in mesh.z_m for value in row]
        + [output[2], source[2], root[2], maximum_deformation_mm, -maximum_deformation_mm]
        + [point[2] for point in module_points],
    )
    limits = []
    for values in coordinates:
        lower = min(values)
        upper = max(values)
        padding = max(0.05 * (upper - lower), 0.5)
        limits.append((lower - padding, upper + padding))

    frame_step = max(1, len(result.time_s) // frame_count)
    frame_indices = list(range(0, len(result.time_s), frame_step))
    if frame_indices[-1] != len(result.time_s) - 1:
        frame_indices.append(len(result.time_s) - 1)

    figure = plt.figure(figsize=(9, 7))
    axis = figure.add_subplot(111, projection='3d')
    surface = Poly3DCollection(mesh_faces_mm(mesh), facecolor='tab:blue', edgecolor='0.35', linewidth=0.15, alpha=0.85)
    axis.add_collection3d(surface)
    axis.add_collection3d(Poly3DCollection(
        pcb_faces, facecolor='seagreen', edgecolor='0.25', linewidth=0.2, alpha=0.65,
    ))
    axis.add_collection3d(Poly3DCollection(
        buzzer_faces, facecolor='tab:orange', edgecolor='0.25', linewidth=0.2, alpha=0.9,
    ))
    output_marker = axis.scatter(*output, color='tab:red', s=35, label='output on wing')
    axis.scatter(*source, color='tab:orange', s=40, label='buzzer opening')
    axis.scatter([], [], [], color='seagreen', marker='s', s=55, label='PCB module')
    axis.quiver(
        *root,
        -body_arrow_length_mm,
        0.0,
        0.0,
        color='0.15',
        linewidth=2.5,
        arrow_length_ratio=0.22,
        label='body side',
    )
    axis.text(
        body_label_x,
        root[1],
        root[2],
        'BODY SIDE',
        color='0.15',
        fontweight='bold',
        horizontalalignment='right',
        verticalalignment='center',
    )
    distance_line, = axis.plot(
        (source[0], output[0]),
        (source[1], output[1]),
        (source[2], output[2]),
        color='0.4',
        linestyle='--',
        linewidth=1.0,
    )
    axis.set_xlim(*limits[0])
    axis.set_ylim(*limits[1])
    axis.set_zlim(*limits[2])
    axis.set_box_aspect(tuple(upper - lower for lower, upper in limits))
    axis.set_xlabel('wing x [mm]', labelpad=9)
    axis.set_ylabel('wing y [mm]', labelpad=9)
    axis.set_zlabel('wing z [mm]', labelpad=8)
    axis.view_init(elev=23, azim=-62)
    axis.legend(loc='upper right')
    figure.subplots_adjust(left=0.04, right=0.91, bottom=0.08, top=0.88)

    def update(frame_number):
        sample_index = frame_indices[frame_number]
        modal_displacements = tuple(
            history[sample_index] for history in result.modal_displacement_by_mode_m
        )
        deformed_mesh = multimode_deformed_wing_surface(
            bundle.wing,
            modal_displacements,
            n_span=51,
            n_chord=11,
            deformation_scale=deformation_scale,
        )
        deformed_output = (
            output[0],
            output[1],
            output[2] + deformation_scale * result.output_displacement_m[sample_index] * 1e3,
        )
        surface.set_verts(mesh_faces_mm(deformed_mesh))
        output_marker._offsets3d = ([deformed_output[0]], [deformed_output[1]], [deformed_output[2]])
        distance_line.set_data_3d(
            (source[0], deformed_output[0]),
            (source[1], deformed_output[1]),
            (source[2], deformed_output[2]),
        )
        axis.set_title(
            f't = {result.time_s[sample_index]:.6g} s, displayed deformation = {deformation_scale:g}x',
            pad=15,
        )
        return surface, output_marker, distance_line

    update(0)
    return FuncAnimation(
        figure,
        update,
        frames=len(frame_indices),
        interval=interval_ms,
        blit=False,
        cache_frame_data=False,
    )

In [ ]:
bundle = load_profile_bundle(PROJECT_ROOT)
result = run_profile_experiment(bundle)
wing_animation = animate_selected_wing_3d(bundle, result, deformation_scale=100.0)
display(HTML(wing_animation.to_jshtml()))
plt.close(wing_animation._fig)